# 18 · One-match check, and the extra numbers in the deck

Does a player's place depend on one big match? This removes every player's single best match, ranks everyone again, and then tests whether a rule that drops such players would have picked **better** lists: on the 52 internal checks (the same splits as notebook 11) and on the outside matches. It also saves the shuffle-test values, the worked ball from the 2023 IPL final and the shrinkage strengths shown in the deck.

Result: Carse and Overton lean on one match, but the rule made the lists worse, so the lists stay as they are.

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

## A · One-match check, shuffle values, worked ball, settings

<details><summary>Technical notes</summary>

P17 - Extra evidence for the v2 deck (local only).  
  
1. Consistency check. Rule fixed on 2026-11-14 BEFORE running it:  
   For every player, remove his single best match: the match with the largest pressure-weighted value (sum of w * v,  
   w = pressure_pct ** (4 * alpha), exactly the rating's own weights) in his own record. Rebuild the ranking for  
   everyone at once with the final settings (shrinkage strengths held at their full-data values, as in the bootstrap).  
   A top-10 pick keeps his place only if he is still inside the top 20 of available players on that ranking.  
   A pick who fails is replaced by the best-placed player outside the top 10 (in final-rank order) who passes the same  
   test. Batters and bowlers get the identical rule.  
   Removing everyone's best match at once keeps the comparison fair (nobody keeps a best match others lose).  
2. Negative control: the 200 shuffled clutch-spread values behind p11's p-values (same seeds), for a histogram.  
3. Worked ball: the last over of the 2023 IPL final (match 21586, chase) through the expected-outcome model.  
4. Counts for the pipeline diagram and the shrinkage strengths k of the final settings.
</details>

In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from rrlib.common import CFG, SEED, load, save_table, save_json, load_json, load_table, get_logger, Timer, excluded_reasons
from rrlib import ratings as RL

log = get_logger("p17_deck_extras")
v = load("values")
comp = load_json("p05_components")
chosen = load_json("p09_summary")["chosen"]
players = load("players").set_index("player_id")
EXCL = set(excluded_reasons())
PCOL = {"bat": "striker_id", "bowl": "bowler_id"}

# ---------------------------------------------------------------- 1. leave-best-match-out (everyone at once)
rows, counts, corrected = [], {}, {}
for side in ("bat", "bowl"):
    cfg = chosen[side]
    met, alpha = cfg.get("value", "v1"), cfg.get("alpha", 1.0)
    col = f"{met}_{side}"
    RL.K_RECORD = {}
    base = RL.build_scores(v, side, cfg, comp)
    kfix = dict(RL.K_RECORD); RL.K_RECORD = None
    sf = RL.side_frame(v, side, met)
    counts[side] = {"players_with_balls": int(sf.pid.nunique()), "rated_min_pressure_balls": int(len(base)),
                    "available": int((~base.index.isin(EXCL)).sum()),
                    "k": {str(k): float(x) for k, x in kfix.items()}}
    sf["wv"] = sf.pw.clip(1e-3, 1) ** (4 * alpha) * sf.val
    pm = sf.groupby(["pid", "match_id"]).wv.sum().reset_index()
    best = pm.loc[pm.groupby("pid").wv.idxmax()].set_index("pid")
    tot_abs = pm.assign(a=pm.wv.abs()).groupby("pid").a.sum()
    tot = pm.groupby("pid").wv.sum()
    nmatch = pm.groupby("pid").match_id.nunique()

    key = pd.MultiIndex.from_arrays([v[PCOL[side]].values, v.match_id.values])
    drop = key.isin(pd.MultiIndex.from_arrays([best.index.values, best.match_id.values]))
    v2 = v.copy()
    v2.loc[drop, col] = np.nan
    RL.K_FIXED = kfix
    lbmo = RL.build_scores(v2, side, cfg, comp)
    RL.K_FIXED = None

    avail_b = base[~base.index.isin(EXCL)]
    avail_l = lbmo[~lbmo.index.isin(EXCL)]
    rb = avail_b.score.rank(ascending=False, method="min")
    rl = avail_l.score.rank(ascending=False, method="min")
    fin = load_table(f"final_top30_{side}").set_index("player_id")
    for pid, r in fin.iterrows():
        rows.append({"side": side, "player_id": pid, "name": r["name"], "final_rank": int(r.final_rank),
                     "point_rank": int(rb.get(pid, np.nan)), "matches": int(nmatch.get(pid, 0)),
                     "best_match_id": int(best.match_id.get(pid)), "best_match_share_abs": float(best.wv.get(pid) / tot_abs.get(pid)),
                     "best_match_share_net": float(best.wv.get(pid) / tot.get(pid)) if tot.get(pid) > 0 else np.nan,
                     "rank_without_best_match": float(rl.get(pid, np.nan)),
                     "still_eligible_without_best": bool(pid in avail_l.index)})
    T = pd.DataFrame([x for x in rows if x["side"] == side]).sort_values("final_rank")
    T["passes"] = T.rank_without_best_match <= 20
    top = T.head(10)
    keep = top[top.passes].final_rank.tolist()
    need = 10 - len(keep)
    subs = T[(T.final_rank > 10) & T.passes].final_rank.head(need).tolist()
    new = sorted(keep + subs)
    corrected[side] = {"failed": top[~top.passes][["final_rank", "name", "rank_without_best_match"]].to_dict("records"),
                       "replacements": T[T.final_rank.isin(subs)][["final_rank", "name", "rank_without_best_match"]].to_dict("records"),
                       "list_after_rule": T.set_index("final_rank").loc[new, "name"].tolist()}
    log.info(f"{side}: rank without best match (final top 30):\n" +
             T[["final_rank", "name", "matches", "best_match_share_abs", "rank_without_best_match", "passes"]].round(3).to_string(index=False))
    log.info(f"{side}: failed {corrected[side]['failed']} | replacements {corrected[side]['replacements']}")

save_table(pd.DataFrame(rows), "consistency_lbmo")
save_json(corrected, "consistency_lbmo_rule")


# ---------------------------------------------------------------- 2. negative-control values (same seeds as p11)
def shuffled(d, seed):
    r = np.random.RandomState(seed)
    d = d.copy()
    d["pres"] = d.groupby("pid").pres.transform(lambda s: r.permutation(s.values))
    return RL.split_half_cov(d)[0]


neg = []
with Timer(log, "negative control values"):
    for side in ("bat", "bowl"):
        d = RL.side_frame(v, side, chosen[side].get("value", "v1"))
        obs = RL.split_half_cov(d)[0]
        null = Parallel(n_jobs=CFG["n_jobs"])(delayed(shuffled)(d, SEED + i) for i in range(200))
        neg += [{"side": side, "kind": "observed", "value": obs}] + [{"side": side, "kind": "shuffled", "value": x} for x in null]
        log.info(f"{side}: observed {obs:.3e}; p = {np.mean(np.array(null) >= obs):.3f}")
save_table(pd.DataFrame(neg), "negative_control_values")

# ---------------------------------------------------------------- 3. worked ball
bs = load("ball_state")
bs = bs[~bs.is_super_over].reset_index(drop=True)
cx = load("ctx_preds")
assert len(bs) == len(v) == len(cx)
m = (bs.match_id == 21586) & (bs.innings == 2) & (bs.over_no == 15)
wb = bs.loc[m, ["match_id", "legal_before", "need", "balls_left", "wkts_before", "runs", "is_four", "is_six", "striker_id", "bowler_id"]].copy()
for c in ["wp_now", "wp_next", "p_dot", "p_1", "p_2", "p_3", "p_4", "p_6", "p_wkt", "p_x", "x_bat_runs"]:
    wb[c] = cx.loc[m, c].values
for c in ["v1_bat", "v3_bat", "v1_bowl", "pressure", "pressure_pct", "LI"]:
    wb[c] = v.loc[m, c].values
wb["striker"] = players.name.reindex(wb.striker_id).values
wb["bowler"] = players.name.reindex(wb.bowler_id).values
save_table(wb, "worked_ball_2023_final")

save_json(counts, "deck_counts")
log.info(f"counts: {counts}")


19:04:31 bat: rank without best match (final top 30):
 final_rank                 name  matches  best_match_share_abs  rank_without_best_match  passes
          1      Abhishek Sharma      134                 0.047                      1.0    True
          2 Vaibhav Sooryavanshi       35                 0.133                      3.0    True
          3     Suryakumar Yadav      139                 0.053                      2.0    True
          4            Phil Salt       84                 0.083                      5.0    True
          5            Tim David       98                 0.108                      7.0    True
          6     Yashasvi Jaiswal       89                 0.061                      4.0    True
          7     Heinrich Klaasen       72                 0.116                      6.0    True
          8          Josh Inglis       56                 0.095                      9.0    True
          9        Rovman Powell      100                 0.074          

19:04:31 bat: failed [] | replacements []


19:04:32 bowl: rank without best match (final top 30):
 final_rank               name  matches  best_match_share_abs  rank_without_best_match  passes
          1     Jasprit Bumrah       75                 0.062                      1.0    True
          2        Lungi Ngidi       39                 0.371                     16.0    True
          3        Rashid Khan      122                 0.052                      2.0    True
          4   Mitchell Santner       83                 0.075                      5.0    True
          5     Benjamin Sears       22                 0.469                     18.0    True
          6     Josh Hazlewood       65                 0.041                      3.0    True
          7       Sunil Narine       67                 0.073                      4.0    True
          8       Nathan Ellis       43                 0.151                      9.0    True
          9      Jamie Overton       30                 0.240                      NaN   F

19:04:32 bowl: failed [{'final_rank': 9, 'name': 'Jamie Overton', 'rank_without_best_match': nan}, {'final_rank': 10, 'name': 'Brydon Carse', 'rank_without_best_match': 61.0}] | replacements [{'final_rank': 12, 'name': 'Kuldeep Yadav', 'rank_without_best_match': 7.0}, {'final_rank': 13, 'name': 'Varun Chakravarthy', 'rank_without_best_match': 6.0}]


19:04:50 bat: observed -7.386e-04; p = 0.710


19:05:06 bowl: observed -8.058e-03; p = 0.805


19:05:06 [negative control values] 34.2s


19:05:07 counts: {'bat': {'players_with_balls': 888, 'rated_min_pressure_balls': 174, 'available': 149, 'k': {"('tau_d', 'bat', 'v3')": 0.0, "('r2', 'bat', 'v3', 'weighted', 0.5)": 967.5077945837668}}, 'bowl': {'players_with_balls': 647, 'rated_min_pressure_balls': 191, 'available': 161, 'k': {"('tau_d', 'bowl', 'v1')": 0.0, "('r2', 'bowl', 'v1', 'weighted', 0.5)": 2400.285416111925}}}


## B · The one-match rule on the outside matches

<details><summary>Technical notes</summary>

P17b - Outside-data check of the consistency rule (local only).  
Scores the original bowling top 10 and the list after the one-match rule (p17) on the 3,993 public matches, with the  
exact yardstick of p14 (runs value per 100 public pressure balls above the pool average, same pool, same groups).  
Also scores each swapped player on his own. The outside data was never used to tune anything.
</details>

In [3]:
import numpy as np
import pandas as pd
from rrlib.common import load, load_table, load_json, save_table, get_logger, excluded_players
from rrlib import ratings as RL

log = get_logger("p17b_external_lists")
v = load("values_ext")
V0 = load("values")
players = load("players").set_index("player_id")
EXCL = excluded_players()
rule = load_json("consistency_lbmo_rule")
GROUPS = {"all public matches": np.ones(len(v), bool), "T20 World Cups": v.is_wc.eq(1).values,
          "franchise leagues": v.level.isin(["BBL", "PSL", "CPL", "SA20", "ILT20", "MLC", "BPL", "LPL", "Blast"]).values,
          "full-member T20Is (incl. World Cups)": v.level.eq("T20I-full").values}
name_to_id = {}
rows = []
for s in ("bat", "bowl"):
    fin = load_table(f"final_top30_{s}").set_index("player_id")
    ids = dict(zip(fin["name"], fin.index))
    lists = {"original top 10": fin.sort_values("final_rank").index[:10].tolist(),
             "after one-match rule": [ids[n] for n in rule[s]["list_after_rule"]]}
    pc = RL.PCOL[s]
    base_ids = set(V0.loc[V0.is_pressure & V0[f"v1_{s}"].notna(), pc].unique())
    for gname, gm in GROUPS.items():
        d = v[gm & v.is_pressure & v[f"wpa_{s}"].notna()]
        y = d.groupby(pc).agg(y=(f"wpa_{s}", "mean"), y1=(f"v1_{s}", "mean"), n=(f"wpa_{s}", "size"))
        y = y[(y.n >= 20) & y.index.isin(players.index) & y.index.isin(base_ids) & ~y.index.isin(EXCL)]
        avg1 = np.average(y.y1, weights=y.n)
        for lname, L in lists.items():
            inp = [i for i in L if i in y.index]
            val = 100 * (np.average(y.loc[inp, "y1"], weights=y.loc[inp, "n"]) - avg1) if inp else np.nan
            rows.append({"side": s, "group": gname, "list": lname, "players_in_pool": len(inp),
                         "runs_per_100_pressure_balls_vs_avg": val})
        if s == "bowl":
            for pid in set(lists["original top 10"]) ^ set(lists["after one-match rule"]):
                rows.append({"side": s, "group": gname, "list": f"single: {players.name.get(pid)}",
                             "players_in_pool": int(pid in y.index),
                             "public_pressure_balls": int(y.n.get(pid, 0)),
                             "runs_per_100_pressure_balls_vs_avg": 100 * (y.y1.get(pid) - avg1) if pid in y.index else np.nan})
R = pd.DataFrame(rows)
save_table(R, "consistency_rule_external")
log.info("\n" + R.round(2).to_string())


19:05:07 
    side                                 group                        list  players_in_pool  runs_per_100_pressure_balls_vs_avg  public_pressure_balls
0    bat                    all public matches             original top 10                9                                2.00                    NaN
1    bat                    all public matches        after one-match rule                9                                2.00                    NaN
2    bat                        T20 World Cups             original top 10                5                               -0.53                    NaN
3    bat                        T20 World Cups        after one-match rule                5                               -0.53                    NaN
4    bat                     franchise leagues             original top 10                5                                2.63                    NaN
5    bat                     franchise leagues        after one-match rule          

## C · The one-match rule on the 52 internal checks

<details><summary>Technical notes</summary>

P17c - Does the one-match (leave-best-match-out) rule pick better top 10s on matches the method never saw?  
  
Same 52 splits and yardstick as P9 (2 rolling-origin + 50 random 50/50 match splits within 2022-25), plus the  
2022-25 -> 2026 split for information. On each split, using training matches only:  
  * plain list: top 10 of the pool by the final settings' score (exactly as P9 evaluates);  
  * rule list : the p17 rule. Remove every player's single best training match (largest sum of w*v), rebuild all  
    scores with the shrinkage strengths held at their training values, rank the pool again; a top-10 pick stays only  
    if still inside the top `thr` (20, as pre-registered in p17), else the next plain-ranked player who passes.  
Both lists are scored on the hidden matches: pressure-ball value above expected (V-2 win prob, V-1 runs),  
n-weighted mean of the top 10 minus the pool mean. Paired difference rule - plain over splits.
</details>

In [4]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from rrlib.common import CFG, SEED, load, save_table, save_json, load_json, get_logger, Timer, weighted_spearman
from rrlib import ratings as RL

log = get_logger("p17c_consistency_cv")
V_ALL = load("values")
comp = load_json("p05_components")
chosen = load_json("p09_summary")["chosen"]
HOLD = CFG["cv"]["holdout_season"]
V = V_ALL[V_ALL.year < HOLD].copy()
MIN_TEST = 20
THRS = (15, 20, 30)

splits = []
for tr_years, te_year in CFG["cv"]["rolling"]:
    splits.append(("rolling", f"{tr_years[0]}-{tr_years[-1]}->{te_year}", V.year.isin(tr_years).values, (V.year == te_year).values))
mids = np.sort(V.match_id.unique())
rng = np.random.RandomState(SEED)
for i in range(CFG["cv"]["random_splits"]):
    a = set(rng.choice(mids, len(mids) // 2, replace=False))
    m = V.match_id.isin(a).values
    splits.append(("random", f"r{i}", m, ~m))


def yardstick(test, side):
    pcol = RL.PCOL[side]
    t = test[test.is_pressure & test[f"v2_{side}"].notna()]
    g = t.groupby(pcol).agg(y2=(f"v2_{side}", "mean"), y1=(f"v1_{side}", "mean"), n=(f"v2_{side}", "size"))
    return g[g.n >= MIN_TEST]


def one_split(kind, name, train, test):
    out = []
    for side in ("bat", "bowl"):
        cfg = chosen[side]
        met, alpha = cfg.get("value", "v1"), cfg.get("alpha", 1.0)
        col, pcol = f"{met}_{side}", RL.PCOL[side]
        y = yardstick(test, side)
        base_ids = set(train.loc[train.is_pressure & train[f"v1_{side}"].notna(), pcol].unique())
        pool = y.index[y.index.isin(base_ids)]
        if len(pool) < 15:
            continue
        RL.K_RECORD = {}
        base = RL.build_scores(train, side, cfg, comp).score
        kfix = dict(RL.K_RECORD); RL.K_RECORD = None
        # best match per player in TRAINING data, with the rating's own ball weights
        sf = RL.side_frame(train, side, met)
        sf["wv"] = sf.pw.clip(1e-3, 1) ** (4 * alpha) * sf.val
        pm = sf.groupby(["pid", "match_id"]).wv.sum().reset_index()
        best = pm.loc[pm.groupby("pid").wv.idxmax()]
        key = pd.MultiIndex.from_arrays([train[pcol].values, train.match_id.values])
        drop = key.isin(pd.MultiIndex.from_arrays([best.pid.values, best.match_id.values]))
        t2 = train.copy()
        t2.loc[drop, col] = np.nan
        RL.K_FIXED = kfix
        lbmo = RL.build_scores(t2, side, cfg, comp).score
        RL.K_FIXED = None

        s = base.reindex(pool).fillna(-1e9).sort_values(ascending=False)
        rl = lbmo.reindex(pool).dropna().rank(ascending=False, method="min")
        yy = y.loc[pool]
        wmean = lambda c, idx: np.average(yy.loc[idx, c], weights=yy.loc[idx, "n"])
        plain = list(s.index[:10])
        sp = base.reindex(pool).fillna(-1e9)
        row = {"kind": kind, "split": name, "side": side, "n_pool": len(pool),
               "plain_rho_v2": weighted_spearman(sp.values, yy.loc[pool].y2.values, yy.loc[pool].n.values),
               "plain_rho_v1": weighted_spearman(sp.values, yy.loc[pool].y1.values, yy.loc[pool].n.values),
               "plain_lift_v2": wmean("y2", plain) - wmean("y2", pool),
               "plain_lift_v1": wmean("y1", plain) - wmean("y1", pool)}
        for thr in THRS:
            passes = [p for p in s.index if rl.get(p, np.inf) <= thr]
            keep = [p for p in plain if p in passes]
            subs = [p for p in s.index[10:] if p in passes][: 10 - len(keep)]
            lst = keep + subs
            row[f"rule{thr}_changed"] = 10 - len(keep)
            row[f"rule{thr}_lift_v2"] = wmean("y2", lst) - wmean("y2", pool)
            row[f"rule{thr}_lift_v1"] = wmean("y1", lst) - wmean("y1", pool)
        out.append(row)
    return out


with Timer(log, "52 splits + 2026"):
    jobs = [(k, n, V[tr], V[te]) for k, n, tr, te in splits]
    jobs.append(("holdout", "2022-25->2026", V_ALL[V_ALL.year < HOLD], V_ALL[V_ALL.year == HOLD]))
    res = Parallel(n_jobs=min(8, CFG["n_jobs"]))(delayed(one_split)(*j) for j in jobs)
R = pd.DataFrame([r for x in res for r in x])
save_table(R, "consistency_rule_cv")

summ = {}
for side in ("bat", "bowl"):
    r = R[(R.side == side) & (R.kind != "holdout")]
    h = R[(R.side == side) & (R.kind == "holdout")]
    summ[side] = {}
    for thr in THRS:
        d = {}
        for y in ("v2", "v1"):
            diff = r[f"rule{thr}_lift_{y}"] - r[f"plain_lift_{y}"]
            ch = r[f"rule{thr}_changed"] > 0
            d[y] = {"plain_mean": float(r[f"plain_lift_{y}"].mean()), "rule_mean": float(r[f"rule{thr}_lift_{y}"].mean()),
                    "mean_diff": float(diff.mean()), "se_diff": float(diff.std(ddof=1) / np.sqrt(len(diff))),
                    "rule_better": int((diff[ch] > 0).sum()), "rule_worse": int((diff[ch] < 0).sum()),
                    "holdout_diff": float((h[f"rule{thr}_lift_{y}"] - h[f"plain_lift_{y}"]).iloc[0]) if len(h) else None}
        d["splits_changed"] = int((r[f"rule{thr}_changed"] > 0).sum())
        d["mean_names_changed"] = float(r[f"rule{thr}_changed"].mean())
        summ[side][thr] = d
        log.info(f"{side} thr {thr}: changed {d['splits_changed']}/{len(r)} splits (mean {d['mean_names_changed']:.2f} names) | "
                 f"V2 diff {d['v2']['mean_diff']:+.5f} (SE {d['v2']['se_diff']:.5f}, better {d['v2']['rule_better']} / worse {d['v2']['rule_worse']}) | "
                 f"V1 diff {d['v1']['mean_diff']:+.4f} (SE {d['v1']['se_diff']:.4f}, better {d['v1']['rule_better']} / worse {d['v1']['rule_worse']}) | "
                 f"2026 V2 {d['v2']['holdout_diff']:+.5f} V1 {d['v1']['holdout_diff']:+.4f}")
save_json(summ, "consistency_rule_cv_summary")


19:05:26 [52 splits + 2026] 18.7s


19:05:26 bat thr 15: changed 40/52 splits (mean 1.27 names) | V2 diff +0.00002 (SE 0.00014, better 19 / worse 21) | V1 diff -0.0052 (SE 0.0039, better 12 / worse 28) | 2026 V2 +0.00000 V1 +0.0000


19:05:26 bat thr 20: changed 34/52 splits (mean 1.10 names) | V2 diff +0.00010 (SE 0.00013, better 17 / worse 17) | V1 diff -0.0027 (SE 0.0034, better 12 / worse 22) | 2026 V2 +0.00000 V1 +0.0000


19:05:26 bat thr 30: changed 34/52 splits (mean 1.08 names) | V2 diff +0.00008 (SE 0.00013, better 17 / worse 17) | V1 diff -0.0016 (SE 0.0034, better 14 / worse 20) | 2026 V2 +0.00000 V1 +0.0000


19:05:26 bowl thr 15: changed 44/52 splits (mean 1.50 names) | V2 diff +0.00006 (SE 0.00013, better 21 / worse 23) | V1 diff -0.0051 (SE 0.0038, better 18 / worse 26) | 2026 V2 -0.00066 V1 -0.0459


19:05:26 bowl thr 20: changed 37/52 splits (mean 0.98 names) | V2 diff +0.00018 (SE 0.00011, better 23 / worse 14) | V1 diff +0.0002 (SE 0.0035, better 17 / worse 20) | 2026 V2 -0.00066 V1 -0.0459


19:05:26 bowl thr 30: changed 29/52 splits (mean 0.77 names) | V2 diff +0.00014 (SE 0.00011, better 18 / worse 11) | V1 diff +0.0017 (SE 0.0032, better 14 / worse 15) | 2026 V2 -0.00066 V1 -0.0459
